# ARR rebuttal experiments: Aya-23-8B steering and probes (submission #4657)

**Runtime:** Runtime → Change runtime type → **A100 GPU**. Run the cells top to bottom.

Everything is written to Google Drive under `MyDrive/detox_rebuttal/`. If Colab disconnects, re-run cells 1–3, then continue from the cell that stopped; finished steps are skipped.

| Cell | Experiment | Approx. A100 time |
|---|---|---|
| 4 | Probe controls, Aya random seeds 42 and 1337 (pair-grouped, LOLO, within-target, by source, dense layers) | ~30 min |
| 5 | Second confirmatory run, adapter random seed 1337: dev selection | ~2 h |
| 6 | Lock the dev selection, then frozen test + controls + bootstrap | ~1.5 h |
| 7 | Wider α grid {3, 4, 8} on dev, adapter random seed 42 (exploratory) | ~1.5 h |
| 8 | Source-subset bootstrap (CPU) | 1 min |
| 9 | Optional: retrain Qwen adapters and rerun probes | ~1 h per random seed |
| 10 | Zip results for upload | 1 min |

Cells 5–7 are independent of cell 4. If time is short, run 4 → 5 → 6 → 8 → 10 first.


In [ ]:
# Cell 1: mount Drive and get the code
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
BASE = '/content/drive/MyDrive/detox_rebuttal'
REPO = f'{BASE}/repo'
os.makedirs(BASE, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'confirmatory-aya-steering-2026-09-30',
                    'https://github.com/farunawebservices/multilingual-detox-mech', REPO], check=True)
os.chdir(REPO)
!ls -la results/training_aya_23_8b/seed_42/adapter/adapter_model.safetensors results/training_aya_23_8b/seed_1337/adapter/adapter_model.safetensors
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Cell 2: install packages and log in to Hugging Face
# Aya-23-8B is gated: accept the licence at https://huggingface.co/CohereLabs/aya-23-8B,
# then add your token in Colab (key icon on the left) as a secret named HF_TOKEN.
!pip -q install -U "transformers>=4.44,<5" "peft>=0.12" "bitsandbytes>=0.43" accelerate sentence-transformers sacrebleu scikit-learn
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))


In [ ]:
# Cell 3: fetch the rebuttal scripts into the repo (written for this rebuttal; nothing else in the repo is modified)
import os, urllib.request
os.chdir(REPO)
os.makedirs('scripts/rebuttal', exist_ok=True)
# Upload the four .py files from the zip I sent you into scripts/rebuttal/ with the Files pane,
# or run this cell after uploading them to /content:
import shutil, glob
for f in glob.glob('/content/*.py'):
    shutil.copy(f, 'scripts/rebuttal/')
!ls scripts/rebuttal


In [ ]:
# Cell 4: probe controls for Aya random seeds 42 and 1337
import os
os.chdir(REPO)
OUT = f'{BASE}/results/probe_controls'
for seed in [42, 1337]:
    if os.path.exists(f'{OUT}/probe_controls_aya_seed{seed}_summary.csv'):
        print('skip seed', seed); continue
    !python scripts/rebuttal/40_probe_controls.py --model aya --seed {seed} --out {OUT}


In [ ]:
# Cell 5: second confirmatory run with the Aya random-seed-1337 adapter: development selection.
# Same preregistered grid as the paper's run: k in {16, 20}, alpha in {0, 0.25, 0.5, 1, 2}.
import os
os.chdir(REPO)
RUN = 'replicate_seed1337'
RUN_DIR = f'artifacts/confirmatory_aya_2026_09_30/{RUN}'
if not os.path.exists(f'{RUN_DIR}/config/dev_selection.json'):
    !python scripts/34_aya_confirmatory_steering.py --seed 1337 --phase dev --layers 16,20 --strengths 0.0,0.25,0.5,1.0,2.0 --run-name {RUN}
!cat {RUN_DIR}/config/dev_selection.json | head -20


In [ ]:
# Cell 6: lock the dev-selected configuration, then run the frozen test, the two extra controls, and the bootstrap.
# The lock is written from dev_selection.json only; the test split is not read before this point.
import json, os
os.chdir(REPO)
RUN = 'replicate_seed1337'
RUN_DIR = f'artifacts/confirmatory_aya_2026_09_30/{RUN}'
sel = json.load(open(f'{RUN_DIR}/config/dev_selection.json'))['selected']
lock = {
    "model_id": "CohereLabs/aya-23-8B", "adapter_seed": 1337,
    "direction_source": "English training rows only", "target_languages": ["xh", "yo"],
    "direction_estimation_split": "train.csv", "selection_split": "dev.csv", "evaluation_split": "test.csv",
    "selected_layer": int(sel["layer"]), "selected_alpha": float(sel["alpha"]),
    "selection_method": "development-only automatic selection (same rule as primary_seed42)",
    "frozen_test_not_used_for_selection": True,
}
lp = f'{RUN_DIR}/config/locked_test_configuration.json'
if not os.path.exists(lp):
    json.dump(lock, open(lp, 'w'), indent=2)
print(open(lp).read())
if not os.path.exists(f'{RUN_DIR}/metrics/test_metrics_summary.csv'):
    !python scripts/34_aya_confirmatory_steering.py --seed 1337 --phase test --layers 16,20 --strengths 0.0,0.25,0.5,1.0,2.0 --run-name {RUN}
if not os.path.exists(f'{RUN_DIR}/test/orthogonal_random_direction_metrics.csv'):
    !RUN_DIR={RUN_DIR} ADAPTER_SEED=1337 python scripts/rebuttal/35p_control_extension.py
!RUN_DIR={RUN_DIR} python scripts/rebuttal/36p_paired_bootstrap.py


In [ ]:
# Cell 7 (exploratory): wider alpha grid on DEV only, adapter random seed 42. Test data is not touched.
import os
os.chdir(REPO)
RUN = 'seed42_wide_alpha_dev'
if not os.path.exists(f'artifacts/confirmatory_aya_2026_09_30/{RUN}/dev/dev_summary.csv'):
    !python scripts/34_aya_confirmatory_steering.py --seed 42 --phase dev --layers 16,20 --strengths 0.0,3.0,4.0,8.0 --run-name {RUN}
import pandas as pd
print(pd.read_csv(f'artifacts/confirmatory_aya_2026_09_30/{RUN}/dev/dev_summary.csv').round(4).to_string(index=False))


In [ ]:
# Cell 8: source-subset bootstrap (original human-written vs AfriHate-derived) for both runs. CPU only.
import os
os.chdir(REPO)
for run in ['primary_seed42', 'replicate_seed1337']:
    d = f'artifacts/confirmatory_aya_2026_09_30/{run}'
    if os.path.exists(f'{d}/test'):
        !python scripts/rebuttal/41_subset_bootstrap.py --run-dir {d}


In [ ]:
# Cell 9 (optional): retrain the Qwen2.5-7B-Instruct adapters (they are not in the repo), then run probe controls.
# Same config as the paper (QLoRA r16, 3 epochs). Roughly 1 A100-hour per random seed.
import os
os.chdir(REPO)
for seed in [42, 1337, 2025]:
    if not os.path.exists(f'results/training_qwen2_5_7b/seed_{seed}/adapter/adapter_model.safetensors'):
        !python scripts/09_qwen2_5_7b_qlora_finetune.py --seed {seed}
    !python scripts/rebuttal/40_probe_controls.py --model qwen --seed {seed} --out {BASE}/results/probe_controls


In [ ]:
# Cell 10: bundle results (no model weights) for upload back to Claude
import os, shutil
os.chdir(REPO)
pack = f'{BASE}/rebuttal_results'
os.makedirs(pack, exist_ok=True)
for run in ['replicate_seed1337', 'seed42_wide_alpha_dev', 'primary_seed42']:
    src = f'artifacts/confirmatory_aya_2026_09_30/{run}'
    if os.path.exists(src):
        shutil.copytree(src, f'{pack}/{run}', dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('*.pt'))
shutil.copytree(f'{BASE}/results/probe_controls', f'{pack}/probe_controls', dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('*.npz'))
shutil.make_archive(f'{BASE}/rebuttal_results', 'zip', pack)
print('Upload this file to Claude:', f'{BASE}/rebuttal_results.zip')
